# 02 Event Study

Purpose: explain whether event-window returns came from company-specific reaction, QQQ beta, or semiconductor sector beta.

Before running this notebook, populate `data/manual/events_template.csv` and run:

```bash
uv run python -m scripts.import_events data/manual/events_template.csv
uv run python -m scripts.build_event_returns
```

In [ ]:
import duckdb

DB = '../data/duckdb/quant_learn.duckdb'
con = duckdb.connect(DB, read_only=True)

In [ ]:
events = con.execute('''
select e.event_date, e.ticker, e.event_type, e.event_name,
       r.return_0_p1, r.return_0_p5, r.return_0_p20,
       r.abnormal_return_0_p5, r.sector_abnormal_return_0_p5,
       r.pre_event_runup_20d, r.post_event_drift_20d
from event_returns r
join events e using (event_id)
order by e.event_date desc, e.ticker
''').fetchdf()
events

In [ ]:
summary = events.groupby(['event_type', 'ticker']).agg(
    n=('event_type', 'size'),
    avg_car_0_5=('return_0_p5', 'mean'),
    avg_abnormal_0_5=('abnormal_return_0_p5', 'mean'),
    avg_sector_abnormal_0_5=('sector_abnormal_return_0_p5', 'mean'),
).reset_index()
summary

Five-sentence event review:

1. What happened?
2. What did the market likely expect beforehand?
3. Which actual metric surprised: revenue, segment growth, margin, CapEx, guidance, or risk?
4. Was the price reaction company alpha, QQQ beta, or SOXX/SMH sector beta?
5. Did this event change the next 1-2 quarter research hypothesis?